# Professional EDA Report: Credit Risk Modeling Dataset

This notebook demonstrates a production-style exploratory data analysis workflow using the repository's credit risk modeling pipeline. It combines dataset validation, feature profiling, target signal analysis, multicollinearity assessment, and baseline model evaluation.

## Objectives

1. Load the project dataset source and validate data integrity.
2. Profile feature groups and descriptive statistics.
3. Quantify target signal and rank candidate predictors.
4. Identify multicollinearity and redundant feature pairs.
5. Assess baseline predictive performance with robust cross-validation.

## Notebook scope and business questions

This analysis is structured to answer the following questions:
- What synthetic credit risk features are available, and how are they generated?
- Which predictors provide the strongest default signal?
- How correlated are the feature groups, and what redundancy exists?
- Which model families perform best in terms of discrimination and calibration?
- What deployment artifacts and governance metadata are necessary for production use?

In [ ]:
import pandas as pd

feature_descriptions = {
    "income": "Estimated annual income for the borrower.",
    "debt_to_income": "Ratio of total debt payments to income.",
    "credit_utilization": "Fraction of available credit currently used.",
    "open_accounts": "Count of active credit accounts.",
    "delinq_30d": "Number of 30-day delinquencies in the recent period.",
    "delinq_90d": "Number of 90-day delinquencies in the recent period.",
    "age": "Borrower age in years.",
    "employment_years": "Years of continuous employment.",
    "loan_amount": "Requested loan amount.",
    "loan_term": "Loan term in months.",
    "recent_inquiries": "Number of recent credit inquiries.",
    "savings_balance": "Reported savings balance at time of application.",
}

pd.DataFrame(
    {"feature": list(feature_descriptions), "description": list(feature_descriptions.values())}
)

In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (ExtraTreesClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
%matplotlib inline

project_path = Path("../projects/machine_learning/credit_risk_modeling/train.py")
spec = importlib.util.spec_from_file_location("credit_risk_modeling", project_path)
project_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_module)

df, target = project_module.make_synthetic_credit_data()
X = df.copy()
X["default"] = target
X.head()

## 1. Data ingestion and validation

Confirm the origin and structure of the dataset, including shape, class balance, duplicates, missing values, and data types.

In [ ]:
print("Dataset source:", project_path)
print("Dataset shape:", X.shape)
print("Class balance:")
print(X["default"].value_counts(normalize=True).rename("proportion"))
print("\nDuplicate rows:", X.duplicated().sum())

missing_summary = X.isna().sum().astype(int)
missing_summary = missing_summary[missing_summary > 0]
print("\nMissing values by column:")
print(missing_summary if not missing_summary.empty else "No missing values detected.")

print("\nData types:")
print(X.dtypes)

## 2. Feature profile and structure

The synthetic credit risk project defines domain-specific financial predictors. We verify feature coverage and summary statistics before modeling.

In [ ]:
feature_cols = [col for col in X.columns if col != "default"]

feature_summary = (
    X[feature_cols]
    .describe()
    .T
    .assign(feature=lambda df: df.index)
    .reset_index(drop=True)
)

feature_summary["feature"] = feature_summary.index.map(lambda i: feature_summary.loc[i, "feature"])
feature_summary.head(12)

## 2.1 Feature group profile and summary

Group features by domain terms and inspect aggregate distribution statistics for each feature family.

In [ ]:
group_labels = [feature.split("_")[0] for feature in feature_cols]

group_stats = []
for group in sorted(set(group_labels)):
    group_features = [feature for feature, label in zip(feature_cols, group_labels) if label == group]
    group_mean = X[group_features].mean().mean()
    group_std = X[group_features].std().mean()
    group_stats.append({"group": group, "group_mean": group_mean, "group_std": group_std})

group_stats_df = pd.DataFrame(group_stats)
group_stats_df

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=group_stats_df, x="group", y="group_mean", color="#4c72b0")
ax.set_title("Average feature value by group")
ax.set_ylabel("Mean value")
ax.set_xlabel("Feature group")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Target signal and predictor ranking

Rank features by their correlation with the default label to determine which inputs carry the strongest predictive signal.

In [ ]:
corr_df = (
    X[feature_cols]
    .corrwith(X["default"])
    .abs()
    .reset_index()
    .rename(columns={0: "abs_target_corr", "index": "feature"})
    .sort_values("abs_target_corr", ascending=False)
)
corr_df.head(10)

## 4. Distributional analysis of top predictors

Compare the distribution of the top-ranked features across default classes. This helps validate whether the selected signals are separable and robust.

In [ ]:
top_features = corr_df.head(6)["feature"].tolist()
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, feature in zip(axes.ravel(), top_features):
    sns.boxplot(x="default", y=feature, data=X, palette="Set2", ax=ax)
    ax.set_title(f"{feature} by Default")
    ax.set_xlabel("Default (0 = no, 1 = yes)")
    ax.set_ylabel(feature)
plt.tight_layout()

## 5. Multicollinearity and redundancy

Identify highly correlated feature pairs that may require dimensionality reduction or feature selection.

In [ ]:
correlation_matrix = X[feature_cols].corr()
high_corr_pairs = (
    correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "corr"})
    .assign(abs_corr=lambda df: df["corr"].abs())
    .query("abs_corr > 0.90")
    .sort_values("abs_corr", ascending=False)
)

print("High correlation pairs (abs > 0.90):")
print(high_corr_pairs.head(12))

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, vmin=-1, vmax=1, annot=False, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix,
                             roc_auc_score)
from sklearn.model_selection import train_test_split

best_model_name = metrics_df.loc[0, "model"]
best_pipeline = pipelines[best_model_name]

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

best_pipeline.fit(X_train, y_train)
y_pred_holdout = best_pipeline.predict(X_test)
y_proba_holdout = best_pipeline.predict_proba(X_test)[:, 1]

print("Best benchmark model:", best_model_name)
print("ROC AUC (holdout):", roc_auc_score(y_test, y_proba_holdout).round(4))
print("Average precision (holdout):", average_precision_score(y_test, y_proba_holdout).round(4))
print("Brier score (holdout):", brier_score_loss(y_test, y_proba_holdout).round(4))
print("\nClassification report:")
print(classification_report(y_test, y_pred_holdout, digits=3))

cm = confusion_matrix(y_test, y_pred_holdout)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion matrix for {best_model_name}")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()

## 6. Baseline predictive signal

Establish baseline model performance using out-of-fold ROC-AUC and compare a regularized linear model to a tree-based ensemble. This is the first quality gate before feature engineering.

In [ ]:
X_features = X[feature_cols]
y = X["default"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipelines = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    "GradientBoosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(random_state=42))
    ]),
    "HistGradientBoosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingClassifier(random_state=42))
    ]),
    "ExtraTrees": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ExtraTreesClassifier(n_estimators=200, random_state=42))
    ])
}

metrics = []
for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_features,
        y,
        cv=cv,
        scoring=["roc_auc", "average_precision", "neg_brier_score"],
        n_jobs=-1,
        return_train_score=False,
    )
    metrics.append(
        {
            "model": name,
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "roc_auc_std": scores["test_roc_auc"].std(),
            "avg_precision_mean": scores["test_average_precision"].mean(),
            "avg_precision_std": scores["test_average_precision"].std(),
            "brier_score_mean": -scores["test_neg_brier_score"].mean(),
            "brier_score_std": scores["test_neg_brier_score"].std(),
        }
    )

metrics_df = pd.DataFrame(metrics).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
metrics_df

plt.figure(figsize=(10, 6))
plot_df = metrics_df.melt(
    id_vars="model",
    value_vars=["roc_auc_mean", "avg_precision_mean"],
    var_name="metric",
    value_name="score",
)
sns.barplot(data=plot_df, x="model", y="score", hue="metric", palette="Set2")
plt.title("Cross-validated model performance")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plot_df = metrics_df.melt(
    id_vars="model",
    value_vars=["roc_auc_mean", "brier_score_mean"],
    var_name="metric",
    value_name="score",
)
plot_df.loc[plot_df["metric"] == "brier_score_mean", "score"] *= -1
sns.barplot(data=plot_df, x="model", y="score", hue="metric", palette=["#4c72b0", "#dd8452"])
plt.title("Model quality comparison: ROC-AUC vs (negative) Brier score")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
## 7. Best benchmark model evaluation on holdout data

Select the top-performing model and confirm its performance on a held-out validation fold before committing to a production candidate.
</VSCode.Cell>
<VSCode.Cell language="python">
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix,
                             roc_auc_score)
from sklearn.model_selection import train_test_split

best_model_name = metrics_df.loc[0, "model"]
best_pipeline = pipelines[best_model_name]

X_train_holdout, X_test_holdout, y_train_holdout, y_test_holdout = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

best_pipeline.fit(X_train_holdout, y_train_holdout)
y_pred_holdout = best_pipeline.predict(X_test_holdout)
y_proba_holdout = best_pipeline.predict_proba(X_test_holdout)[:, 1]

print("Best benchmark model:", best_model_name)
print("ROC AUC (holdout):", roc_auc_score(y_test_holdout, y_proba_holdout).round(4))
print("Average precision (holdout):", average_precision_score(y_test_holdout, y_proba_holdout).round(4))
print("Brier score (holdout):", brier_score_loss(y_test_holdout, y_proba_holdout).round(4))
print("\nClassification report:")
print(classification_report(y_test_holdout, y_pred_holdout, digits=3))

cm = confusion_matrix(y_test_holdout, y_pred_holdout)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion matrix for {best_model_name}")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()


## 7. Reproducible preprocessing and feature selection

Build a deterministic preprocessing pipeline and select the strongest predictors using statistical scoring.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (brier_score_loss, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=8)),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

pipeline.fit(X_train, y_train)
selected_mask = pipeline.named_steps["selector"].get_support()
selected_features = [f for f, keep in zip(feature_cols, selected_mask) if keep]
selected_features

## 8. Model calibration and decision thresholds

Evaluate the trained model for calibration and identify a working decision threshold based on test-set performance.

In [ ]:
proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

thresholds = np.linspace(0.01, 0.99, 99)
threshold_metrics = []
for threshold in thresholds:
    y_thresh = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_thresh).ravel()
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    threshold_metrics.append((threshold, precision, recall, f1))

threshold_df = pd.DataFrame(threshold_metrics, columns=["threshold", "precision", "recall", "f1"])
best_threshold = threshold_df.loc[threshold_df["f1"].idxmax(), "threshold"]

print("Best operating threshold (max F1):", best_threshold)
print(threshold_df.sort_values("f1", ascending=False).head(5))

print("\nClassification report at best threshold:")
print(classification_report(y_test, (proba >= best_threshold).astype(int), digits=3))
print("Brier score:", brier_score_loss(y_test, proba).round(4))
print("ROC AUC:", roc_auc_score(y_test, proba).round(4))

prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10)
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker="o", label="LogisticRegression")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration plot")
plt.legend()
plt.grid(True)
plt.show()

## 9. Transformation metadata and operational readiness

Document feature metadata, selection status, and preprocessing steps to support reproducibility and model governance.

In [ ]:
feature_metadata = pd.DataFrame(
    {
        "feature": feature_cols,
        "dtype": X[feature_cols].dtypes.astype(str),
        "missing_pct": X[feature_cols].isna().mean() * 100,
        "abs_target_corr": corr_df.set_index("feature")["abs_target_corr"].loc[feature_cols].values,
        "selected_by_kbest": [f in selected_features for f in feature_cols],
    }
)
feature_metadata.sort_values(["selected_by_kbest", "abs_target_corr"], ascending=[False, False]).reset_index(drop=True)

## 10. Key findings and next steps

- Implemented a deterministic preprocessing pipeline with median imputation, scaling, and statistical feature selection.
- Selected the strongest predictors using ANOVA F-score, which reduces dimensionality and improves interpretability.
- Determined an operational classification threshold using test-set F1 optimization, and confirmed calibration with a probability reliability plot.
- Captured transformation metadata for each feature, including selection status and target correlation.

Next steps for production readiness:
- Persist the validated preprocessing pipeline and selected feature list as part of the model artifact.
- Add dataset versioning and feature registry metadata to support repeatable credit risk scoring.
- Extend the calibration analysis to class-weighted or cost-sensitive thresholds aligned with business risk tolerance.

## 11. Persist artifacts and governance metadata

Save the validated pipeline, selected feature list, and metadata as reusable artifacts for deployment and audit.

In [ ]:
from pathlib import Path
from joblib import dump

artifact_dir = Path("../artifacts/credit_risk_model")
artifact_dir.mkdir(parents=True, exist_ok=True)

pipeline_path = artifact_dir / "credit_risk_pipeline.joblib"
feature_list_path = artifact_dir / "selected_features.csv"
metadata_path = artifact_dir / "feature_metadata.csv"

# Persist the preprocessing/model pipeline and feature metadata for production reuse
dump(pipeline, pipeline_path)
pd.DataFrame({"selected_feature": selected_features}).to_csv(feature_list_path, index=False)
feature_metadata.to_csv(metadata_path, index=False)

print("Saved pipeline to:", pipeline_path)
print("Saved selected features to:", feature_list_path)
print("Saved feature metadata to:", metadata_path)

## 12. Business-aligned thresholding and deployment readiness

Capture the threshold and calibration settings that should be used in production, and log the business-specific tolerances that inform them.

In [ ]:
deployment_metadata = {
    "pipeline_artifact": str(pipeline_path),
    "selected_feature_count": len(selected_features),
    "selected_features": selected_features,
    "best_threshold": float(best_threshold),
    "roc_auc": float(roc_auc_score(y_test, proba)),
    "brier_score": float(brier_score_loss(y_test, proba)),
    "class_balance_0": float((y == 0).mean()),
    "class_balance_1": float((y == 1).mean()),
    "notes": (
        "Use the threshold with maximum F1 as a starting point. "
        "Evaluate cost-sensitive losses before final deployment."
    ),
}

deployment_metadata_path = artifact_dir / "deployment_metadata.json"
import json
with open(deployment_metadata_path, "w", encoding="utf-8") as handle:
    json.dump(deployment_metadata, handle, indent=2)

print("Saved deployment metadata to:", deployment_metadata_path)

## 13. Operational monitoring and next governance steps

- Validate dataset schema and distribution drift on incoming credit data.
- Periodically re-run calibration and threshold analysis against new validation cohorts.
- Track selected feature stability and update feature metadata when feature definitions change.
- Store artifact versioning and dataset fingerprints alongside model metrics for auditability.

## 14. Cost-sensitive threshold optimization

Evaluate thresholds against business-specific cost weights for false positives and false negatives to support credit risk policy.

In [ ]:
# Example cost matrix weights; adjust to business requirements
cost_fp = 1.0  # cost of classifying a non-default as default
cost_fn = 5.0  # cost of classifying a default as non-default

cost_metrics = []
for threshold in thresholds:
    y_thresh = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_thresh).ravel()
    cost = fp * cost_fp + fn * cost_fn
    cost_metrics.append((threshold, cost))

cost_df = pd.DataFrame(cost_metrics, columns=["threshold", "cost"])
best_cost_threshold = cost_df.loc[cost_df["cost"].idxmin(), "threshold"]
print("Best threshold for cost-weighted loss:", best_cost_threshold)
print(cost_df.sort_values("cost").head(5))

## 15. Feature importance and explainability

Record linear model coefficients and feature importance to support interpretation and feature governance.

In [ ]:
selected_feature_names = [f for f in feature_cols if f in selected_features]
coefficients = pd.Series(
    pipeline.named_steps["model"].coef_[0],
    index=selected_feature_names,
).sort_values(key=lambda x: x.abs(), ascending=False)

feature_importance = pd.DataFrame(
    {
        "feature": selected_feature_names,
        "coefficient": coefficients.values,
        "abs_coefficient": coefficients.abs().values,
    }
).sort_values("abs_coefficient", ascending=False)

feature_importance

## 16. Final recommended implementation plan

1. Persist the validated preprocessing and model artifact along with the selected feature list.
2. Store dataset fingerprint metadata and configuration in the deployment manifest.
3. Define a business-ready threshold using cost-sensitive loss, and lock it into the scoring pipeline.
4. Enable drift detection and yearly recalibration for both feature distributions and model calibration.
5. Keep the feature metadata registry updated with any new feature definitions or transformations.